# Tin tức (Unstructured Data) — One-layer Ingestion

Notebook điều khiển luồng ingestion 1 tầng cho 3 nguồn `vnstock`, `rss`, `html`.

Mỗi nguồn ghi output riêng theo layout:
- `news/<source>/date=<run_date>/PART-000.parquet`
- `news/<source>/date=<run_date>/PART-000.csv`

Schema của cả 3 output là thống nhất theo `NEWS_COLUMNS`, phục vụ downstream sentiment analysis.

In [1]:
import os, sys, warnings, threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook
def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass
    else:
        _orig_hook(args)
threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

from pathlib import Path
root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ingestion.common import configure_logging, load_dotenv_from_project_root
from ingestion.unstructured_data import (
    NEWS_COLUMNS,
    NewsIngestionConfig,
    ingest_news,
    validate_news_df,
)

configure_logging()
load_dotenv_from_project_root()
print("[OK] setup")

📦 **Vnstock 3.5.1 is available**
Current: 3.5.0 (Python 3.12 (venv))
Update: `C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\venv\Scripts\python.exe -m pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

📦 **Vnai 2.4.6 is available**
Current: 2.4.0 (Python 3.12 (venv))
Update: `C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\venv\Scripts\python.exe -m pip install vnai --upgrade`
Release: https://pypi.org/project/vnai/#history

2026-04-17 14:59:48 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env


[OK] setup


## Cấu hình

In [2]:
news_cfg = NewsIngestionConfig(
    use_listing_tickers=True,
    listing_exchange_filter=["HSX", "HNX"],
    max_tickers_per_run=50,
    max_articles_per_source=200,
    days_back=0,
    enable_vnstock=True,
    enable_rss=True,
    enable_html=True,
)

print(f"run_date     : {news_cfg.run_date}")
print(f"news_root    : {news_cfg.news_root}")
print(f"sources.yaml : {news_cfg.resolved_sources_yaml()}")
print(f"listing      : {news_cfg.resolved_listing_parquet()}")
print(f"schema       : {NEWS_COLUMNS}")

run_date     : 2026-04-17
news_root    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news
sources.yaml : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\ingestion\unstructured_data\sources.yaml
listing      : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet
schema       : ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


## Chạy

In [3]:
news_paths = ingest_news(news_cfg)
news_paths

2026-04-17 15:01:31 [INFO] ingest_news[vnstock]: 0 rows
2026-04-17 15:01:31 [INFO] ingest_news[rss]: 60 rows
2026-04-17 15:01:31 [INFO] ingest_news[html]: 47 rows


{'row_counts': {'vnstock': 0, 'rss': 60, 'html': 47},
 'vnstock': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\vnstock\\date=2026-04-17\\PART-000.parquet',
  'csv': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\vnstock\\date=2026-04-17\\PART-000.csv'},
 'rss': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\rss\\date=2026-04-17\\PART-000.parquet',
  'csv': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\rss\\date=2026-04-17\\PART-000.csv'},
 'html': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\html\\date=2026-04-17\\PART-000.parquet',
  'csv': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\html\\date=2026-04-17\\PART-000.csv'}}

## Kiểm tra kết quả

In [4]:
import pandas as pd

print("=" * 70)
print("OUTPUT FILES")
print("=" * 70)
for key in ["vnstock", "rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    csv_path = info.get("csv", "")
    if not parquet_path:
        print(f"  {key:8s}: (trống)")
        continue
    df = pd.read_parquet(parquet_path)
    print(f"  {key:8s}: {len(df):>5d} dòng")
    print(f"    parquet: {parquet_path}")
    print(f"    csv    : {csv_path}")

print(f"\nRow counts: {news_paths.get('row_counts', {})}")

OUTPUT FILES
  vnstock :     0 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\vnstock\date=2026-04-17\PART-000.parquet
    csv    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\vnstock\date=2026-04-17\PART-000.csv
  rss     :    60 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\rss\date=2026-04-17\PART-000.parquet
    csv    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\rss\date=2026-04-17\PART-000.csv
  html    :    47 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\html\date=2026-04-17\PART-000.parquet
    csv    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\html\date=2026-04-17\PART-000.csv

Row counts: {'vnstock': 0, 'rss': 60, 'html': 47}


## Validation & Data Quality

In [5]:
# Validate per-source output with NEWS_COLUMNS
for key in ["vnstock", "rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        print(f"{key}: no output")
        continue

    df = pd.read_parquet(parquet_path)
    issues = validate_news_df(df)
    print(f"{key}: {len(df)} rows")
    if issues:
        print("  ⚠ issues:")
        for i in issues:
            print(f"    - {i}")
    else:
        print("  ✓ validation passed")
    print(f"  columns: {list(df.columns)}")

vnstock: 0 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']
rss: 60 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']
html: 47 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


In [6]:
# Quick preview for sentiment-ready fields
for key in ["vnstock", "rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        continue
    df = pd.read_parquet(parquet_path)
    print(f"\n[{key}] preview")
    print(df[["article_id", "source", "ticker", "title", "body_text", "published_at"]].head(3))


[vnstock] preview
Empty DataFrame
Columns: [article_id, source, ticker, title, body_text, published_at]
Index: []

[rss] preview
                                          article_id             source  \
0  069bb6fe7d377cb89171aaa51c4a6e4c48cdf4bd9a231c...  rss_vnexpress_net   
1  298e0d152858650d35ad28b50b21371cf184eef18d90f6...  rss_vnexpress_net   
2  f2f208ac06523399e0516c13aa3650e40c659e50597c0d...  rss_vnexpress_net   

  ticker                                              title body_text  \
0   None  Bầu Đức: 'Không thiếu tiền khi đầu tư 20.000 h...             
1   None  Châu Âu có thể hết nhiên liệu máy bay trong 6 ...             
2   None                 Highlands Coffee lãi nghìn tỷ đồng             

           published_at  
0  2026-04-17T05:56:24Z  
1  2026-04-17T04:45:00Z  
2  2026-04-17T03:32:58Z  

[html] preview
                                          article_id          source ticker  \
0  7436f22125cebd7b6d13e52be89dc46d1c7c5e5a0e7999...  html_vnexpress   None  